In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

ruta = "/content/drive/MyDrive/TFM"

for raiz, carpetas, archivos in os.walk(ruta):
    for archivo in archivos:
        if "modelado" in archivo.lower():
            print(os.path.join(raiz, archivo))

/content/drive/MyDrive/TFM/dataset_modelado_v1.csv


In [ ]:
ruta_dataset = "/content/drive/MyDrive/TFM/dataset_modelado_v1.csv"

df = pd.read_csv(
    ruta_dataset,
    sep=";",
    encoding="utf-8-sig"
)

df.head()

,periodo,consumo_total_m3,consumo_medio_m3,consumo_medio_diario_m3,abonados,domesticos,industriales,incidencias_sin_lectura,incidencias_normales,codigo_periodo,anio,bimestre,pernoctaciones,consumo_por_abonado,consumo_lag_1,consumo_lag_2,temp_max_media_bimestre,temp_media_bimestre
0,221 ENE-FEB/22,243489.0,24.242234,0.398167,10044,0,0,463,4567,221,2022,1,127192,24.242234,NaN,NaN,24.045763,18.271351
1,222 MAR-ABR/22,243635.0,24.225415,0.392755,10057,0,0,403,4505,222,2022,2,144666,24.225415,243489.0,NaN,23.583607,18.661957
2,223 MAY-JUN/22,283579.0,27.069397,0.437469,10476,0,0,413,4516,223,2022,3,119350,27.069397,243635.0,243489.0,26.519672,21.520070
3,224 JUL-AGO/22,269680.0,26.708923,0.449856,10097,0,0,411,4569,224,2022,4,262985,26.708923,283579.0,243635.0,28.980645,23.612840
4,225 SEP-OCT/22,250460.0,24.722140,0.411999,10131,0,0,367,4498,225,2022,5,170722,24.722140,269680.0,283579.0,28.770492,23.053700


In [ ]:
df.shape

(26, 18)

In [ ]:
df_model = df.dropna().copy()

print(df_model.shape)

(24, 18)


In [ ]:
variables_modelo = [
    "abonados",
    "pernoctaciones",
    "temp_media_bimestre",
    "consumo_lag_1",
    "consumo_lag_2"
]

objetivo = "consumo_total_m3"

X = df_model[variables_modelo]

y = df_model[objetivo]

print(X.shape)
print(y.shape)

(24, 5)
(24,)


In [ ]:
train_size = int(len(df_model) * 0.8)

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]

y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]

periodos_test = df_model["periodo"].iloc[train_size:]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print("\nPeriodos de prueba:")
print(periodos_test.tolist())

Train: (19, 5)
Test: (5, 5)

Periodos de prueba:
['254 JUL-AGO/25', '255 SEP-OCT/25', '256 NOV-DIC /25', '261 ENE-FEB/26', '262 MAR-ABR/26']


¿Es realmente mejor nuestro modelo que una predicción trivial?

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

# Predicción = consumo del bimestre anterior

y_pred_persistencia = X_test["consumo_lag_1"]

mae = mean_absolute_error(
    y_test,
    y_pred_persistencia
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_persistencia
    )
)

mape = np.mean(
    np.abs(
        (y_test - y_pred_persistencia)
        / y_test
    )
) * 100

r2 = r2_score(
    y_test,
    y_pred_persistencia
)

print("MAE:", round(mae,2))
print("RMSE:", round(rmse,2))
print("MAPE:", round(mape,2))
print("R2:", round(r2,4))

MAE: 15710.0
RMSE: 23922.56
MAPE: 4.94
R2: -0.9681


In [ ]:
comparacion = pd.DataFrame({
    "Periodo": periodos_test.values,
    "Real": y_test.values,
    "Persistencia": y_pred_persistencia.values
})

comparacion

,Periodo,Real,Persistencia
0,254 JUL-AGO/25,302485.0,309170.0
1,255 SEP-OCT/25,295781.0,302485.0
2,256 NOV-DIC /25,296665.0,295781.0
3,261 ENE-FEB/26,283305.0,296665.0
4,262 MAR-ABR/26,334222.0,283305.0


Regresión lineal

In [ ]:
from sklearn.linear_model import LinearRegression

modelo_lr = LinearRegression()

modelo_lr.fit(
    X_train,
    y_train
)

y_pred_lr = modelo_lr.predict(X_test)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred_lr
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_lr
    )
)

mape = np.mean(
    np.abs(
        (y_test - y_pred_lr)
        / y_test
    )
) * 100

r2 = r2_score(
    y_test,
    y_pred_lr
)

print("MAE:", round(mae,2))
print("RMSE:", round(rmse,2))
print("MAPE:", round(mape,2))
print("R2:", round(r2,4))

MAE: 9782.1
RMSE: 10939.0
MAPE: 3.22
R2: 0.5885


In [ ]:
comparacion_lr = pd.DataFrame({
    "Periodo": periodos_test.values,
    "Real": y_test.values,
    "Pred_Lineal": y_pred_lr
})

comparacion_lr

,Periodo,Real,Pred_Lineal
0,254 JUL-AGO/25,302485.0,295983.703888
1,255 SEP-OCT/25,295781.0,291894.435357
2,256 NOV-DIC /25,296665.0,278858.959663
3,261 ENE-FEB/26,283305.0,275102.262396
4,262 MAR-ABR/26,334222.0,321708.147728


Conclusión: la incorporación de variables explicativas externas permite mejorar significativamente la capacidad predictiva respecto a un modelo de persistencia basado únicamente en el consumo histórico.

RANDOM FOREST

In [ ]:
from sklearn.ensemble import RandomForestRegressor

modelo_rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=3,
    random_state=42
)

modelo_rf.fit(
    X_train,
    y_train
)

y_pred_rf = modelo_rf.predict(X_test)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred_rf
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_rf
    )
)

mape = np.mean(
    np.abs(
        (y_test - y_pred_rf)
        / y_test
    )
) * 100

r2 = r2_score(
    y_test,
    y_pred_rf
)

print("MAE:", round(mae,2))
print("RMSE:", round(rmse,2))
print("MAPE:", round(mape,2))
print("R2:", round(r2,4))

MAE: 14629.88
RMSE: 17019.4
MAPE: 4.7
R2: 0.0039


In [ ]:
comparacion_rf = pd.DataFrame({
    "Periodo": periodos_test.values,
    "Real": y_test.values,
    "Pred_RF": y_pred_rf
})

comparacion_rf

,Periodo,Real,Pred_RF
0,254 JUL-AGO/25,302485.0,286590.264048
1,255 SEP-OCT/25,295781.0,285108.721530
2,256 NOV-DIC /25,296665.0,282062.751753
3,261 ENE-FEB/26,283305.0,280678.454394
4,262 MAR-ABR/26,334222.0,304868.388911


Conclusión: para horizontes bimestrales y conjuntos de datos de tamaño reducido, los modelos lineales presentan un comportamiento más robusto que los algoritmos ensemble basados en árboles.

XGBoost

In [ ]:
from xgboost import XGBRegressor

modelo_xgb = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    objective="reg:squarederror",
    random_state=42
)

modelo_xgb.fit(
    X_train,
    y_train
)

y_pred_xgb = modelo_xgb.predict(X_test)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred_xgb
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_xgb
    )
)

mape = np.mean(
    np.abs(
        (y_test - y_pred_xgb)
        / y_test
    )
) * 100

r2 = r2_score(
    y_test,
    y_pred_xgb
)

print("MAE:", round(mae,2))
print("RMSE:", round(rmse,2))
print("MAPE:", round(mape,2))
print("R2:", round(r2,4))

MAE: 34338.59
RMSE: 35089.93
MAPE: 11.31
R2: -3.2344


In [ ]:
comparacion_xgb = pd.DataFrame({
    "Periodo": periodos_test.values,
    "Real": y_test.values,
    "Pred_XGB": y_pred_xgb
})

comparacion_xgb

,Periodo,Real,Pred_XGB
0,254 JUL-AGO/25,302485.0,260911.359375
1,255 SEP-OCT/25,295781.0,259086.312500
2,256 NOV-DIC /25,296665.0,261052.078125
3,261 ENE-FEB/26,283305.0,262833.406250
4,262 MAR-ABR/26,334222.0,296881.906250


Conclusión: los modelos avanzados de Machine Learning mejorarán las predicciones respecto a modelos más simples.

En escenarios con series temporales de baja frecuencia y reducido número de observaciones, los modelos lineales pueden presentar un rendimiento superior al de algoritmos de aprendizaje automático más complejos.

¿Qué variables son las que explican la demanda?

In [ ]:
coeficientes = pd.DataFrame({
    "Variable": X_train.columns,
    "Coeficiente": modelo_lr.coef_
})

coeficientes["Abs"] = abs(coeficientes["Coeficiente"])

coeficientes.sort_values(
    "Abs",
    ascending=False
)

,Variable,Coeficiente,Abs
2,temp_media_bimestre,3897.122126,3897.122126
0,abonados,96.948184,96.948184
1,pernoctaciones,0.087582,0.087582
3,consumo_lag_1,-0.064626,0.064626
4,consumo_lag_2,-0.008597,0.008597


In [ ]:
print("Intercepto:")
print(modelo_lr.intercept_)

Intercepto:
-807236.0421104534


Estandarizadas las variables:

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

scaler = StandardScaler()

X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

modelo_std = LinearRegression()

modelo_std.fit(
    X_train_std,
    y_train
)

coef_std = pd.DataFrame({
    "Variable": X_train.columns,
    "Coeficiente_estandarizado": modelo_std.coef_
})

coef_std["Abs"] = abs(
    coef_std["Coeficiente_estandarizado"]
)

coef_std.sort_values(
    "Abs",
    ascending=False
)

,Variable,Coeficiente_estandarizado,Abs
0,abonados,20243.790446,20243.790446
2,temp_media_bimestre,8033.854926,8033.854926
1,pernoctaciones,3935.175769,3935.175769
3,consumo_lag_1,-1602.069356,1602.069356
4,consumo_lag_2,-234.373001,234.373001


La demanda de Guía de Isora no se comporta como una serie temporal “pura”, sino como un fenómeno impulsado principalmente por variables explicativas externas.

Las redes neuronales recurrentes de tipo Long Short-Term Memory (LSTM) constituyen una de las arquitecturas más utilizadas para la predicción de series temporales debido a su capacidad para capturar dependencias temporales complejas y relaciones no lineales entre observaciones consecutivas. Sin embargo, su aplicación requiere habitualmente conjuntos de datos de tamaño suficiente que permitan entrenar un elevado número de parámetros sin incurrir en problemas de sobreajuste. En el presente trabajo, tras el proceso de agregación y depuración de los datos disponibles, el conjunto de modelado quedó constituido por únicamente 24 observaciones bimestrales válidas para entrenamiento y evaluación. Este tamaño muestral resulta insuficiente para explotar adecuadamente las capacidades de aprendizaje de una arquitectura LSTM, pudiendo conducir a modelos inestables y con escasa capacidad de generalización. Además, los resultados obtenidos mediante regresión lineal mostraron un rendimiento significativamente superior al alcanzado por modelos más complejos basados en árboles de decisión, como Random Forest y XGBoost, alcanzando un error porcentual medio absoluto (MAPE) del 3,22 %. Este comportamiento sugiere que la relación entre las variables explicativas y la demanda de agua presenta una estructura predominantemente lineal para la escala temporal considerada. Por tanto, la implementación de una red LSTM no se considera metodológicamente justificada en esta fase del estudio, ya que incrementaría notablemente la complejidad computacional sin garantías razonables de mejora en la capacidad predictiva del modelo.